# 01 — EIA 860 candidate site universe

Builds the candidate-plant table for the PJM data center siting project from EIA Form 860 (2024 release).

Filter chain: all U.S. plants → PJM balancing authority → transmission-class grid voltages (138 kV and above). Result is the ~480-plant candidate universe, cleaned for type consistency and saved to `data/processed/candidates_860.parquet`.

This notebook is self-contained and reproducible: Restart + Run All rebuilds the table and rewrites the parquet, with no network or API dependency. The PJM API work lives separately in `src/pjm_siting/pjm_api.py`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve the repo root by walking up until we find the folder that
# contains both `data` and `src`. This makes the notebook work no matter
# what the kernel's working directory is (project root or notebooks/).
ROOT = Path.cwd()
while not ((ROOT / "data").is_dir() and (ROOT / "src").is_dir()) and ROOT != ROOT.parent:
    ROOT = ROOT.parent

print("pandas", pd.__version__)
print("repo root:", ROOT)
assert (ROOT / "data").is_dir(), "could not locate repo root containing data/"


## Load the raw plant table

In [ ]:
eia_path = ROOT / "data" / "raw" / "eia860" / "2024" / "2___Plant_Y2024.xlsx"
assert eia_path.exists(), f"missing EIA file: {eia_path}"

# skiprows=1 drops the EIA banner row above the real header
plants = pd.read_excel(eia_path, skiprows=1)
print("all plants:", plants.shape)


## Filter to the PJM footprint

In [ ]:
pjm_plants = plants[plants["Balancing Authority Code"] == "PJM"]
print("PJM-footprint plants:", pjm_plants.shape)


In [ ]:
# quick descriptive look at the footprint before the voltage cut
print(pjm_plants["State"].value_counts())


## Filter to transmission-class voltages

The candidate set is plants interconnecting at standard transmission voltage classes. This is a discrete-class filter, not a `>= 138` threshold, so non-standard voltages are excluded by design. Revisit this list if a site you expect is missing.

In [ ]:
transmission_voltages = [138, 230, 345, 500, 765]
pjm_transmission = pjm_plants[pjm_plants["Grid Voltage (kV)"].isin(transmission_voltages)]
print("candidate plants (138 kV and above):", pjm_transmission.shape)


In [ ]:
# state-by-voltage stratification matrix (descriptive deliverable)
pjm_transmission.groupby(["State", "Grid Voltage (kV)"]).size().unstack(fill_value=0)


## Clean column types for parquet

Raw EIA columns are pandas `object` dtype with stray blank cells in otherwise-numeric fields (e.g. a single space in `Zip`). Parquet requires one resolved type per column. We turn whitespace-only cells into real nulls, coerce coordinates to numeric, and cast remaining object columns to string. ZIP stays string on purpose: it is an identifier, not a quantity, and casting to int would strip leading zeros.

In [ ]:
candidates = pjm_transmission.copy()

# whitespace-only cells -> real nulls
candidates = candidates.replace(r"^\s*$", np.nan, regex=True)

# coordinates must be numeric for the later plant-to-pnode proximity mapping
for col in ["Latitude", "Longitude"]:
    candidates[col] = pd.to_numeric(candidates[col], errors="coerce")

# remaining object columns -> clean nullable string
obj_cols = candidates.select_dtypes(include="object").columns
candidates[obj_cols] = candidates[obj_cols].astype("string")

print("dtype summary:")
print(candidates.dtypes.value_counts())


## Validate before saving

In [ ]:
assert candidates.shape[0] == 480, f"expected 480 rows, got {candidates.shape[0]}"
print("rows:", candidates.shape[0], "| cols:", candidates.shape[1])

# coordinate nulls: a few are fine, a flood means a coordinate column held junk
print("\ncoordinate nulls:")
print(candidates[["Latitude", "Longitude"]].isna().sum())


## Save the candidate table

In [ ]:
out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "candidates_860.parquet"

candidates.to_parquet(out_path, index=False)
print("saved", candidates.shape, "to", out_path)
